#### 这段代码首先定义了一个简单的线性模型，然后使用 torch.jit.trace() 和 torch.jit.script() 来优化它。接着，代码测试了原始模型、追踪后的模型和脚本化后的模型，并比较了它们的执行时间。

- 请注意，为了看到显著的性能差异，通常需要在更复杂的模型和更大的输入数据上运行这些优化。此外，torch.jit.trace() 和 torch.jit.script() 的效果可能会因具体的模型和输入数据而异。

- <b>Torch.jit.trace()</b>：这个函数用于追踪 PyTorch 模型的前向传播过程，生成一个 TorchScript 模块。这个模块可以被序列化，从而提高执行速度。

- <b>Torch.jit.script()</b>：这个函数用于将 PyTorch 模型转换为 TorchScript 模块。与 trace() 不同的是，script() 支持更复杂的控制流，包括循环和条件语句。
- TorchScript 适用于静态图，而 JIT 适用于动态图

In [1]:
import torch
import torch.nn as nn

# 定义一个简单的模型
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.linear = nn.Linear(10, 5)

    def forward(self, x):
        return self.linear(x)

# 实例化模型
model = SimpleModel()

# 使用jit.trace()进行追踪
example_input = torch.rand(1, 10)  # 创建一个示例输入
traced_model = torch.jit.trace(model, example_input)

# 使用jit.script()进行脚本化
scripted_model = torch.jit.script(model)

# 测试原始模型
def test_model(model, input_data):
    return model(input_data)

# 测试trace后的模型
def test_traced_model(traced_model, input_data):
    return traced_model(input_data)

# 测试scripted后的模型
def test_scripted_model(scripted_model, input_data):
    return scripted_model(input_data)

# 打印结果
print("Original Model Output:", test_model(model, example_input))
print("Traced Model Output:", test_traced_model(traced_model, example_input))
print("Scripted Model Output:", test_scripted_model(scripted_model, example_input))

# 比较执行时间
import timeit

# 原始模型执行时间
start_time = timeit.default_timer()
for _ in range(100):
    test_model(model, example_input)
end_time = timeit.default_timer()
print(f"Original Model Time: {end_time - start_time:.2f} seconds")

# Traced模型执行时间
start_time = timeit.default_timer()
for _ in range(100):
    test_traced_model(traced_model, example_input)
end_time = timeit.default_timer()
print(f"Traced Model Time: {end_time - start_time:.2f} seconds")

# Scripted模型执行时间
start_time = timeit.default_timer()
for _ in range(100):
    test_scripted_model(scripted_model, example_input)
end_time = timeit.default_timer()
print(f"Scripted Model Time: {end_time - start_time:.2f} seconds")

Original Model Output: tensor([[-0.5392,  0.5904,  0.0317, -0.4259,  0.1070]],
       grad_fn=<AddmmBackward0>)
Traced Model Output: tensor([[-0.5392,  0.5904,  0.0317, -0.4259,  0.1070]],
       grad_fn=<AddmmBackward0>)
Scripted Model Output: tensor([[-0.5392,  0.5904,  0.0317, -0.4259,  0.1070]],
       grad_fn=<AddmmBackward0>)
Original Model Time: 0.00 seconds
Traced Model Time: 0.00 seconds
Scripted Model Time: 0.01 seconds
